In [34]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [35]:
import random
import re
import torch
import torch.nn as nn

from torch.optim import Adam
from torch.utils.data import DataLoader
from collections import Counter

In [36]:
random.seed(33)
np.random.seed(33)
torch.manual_seed(33)

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(33)

In [37]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [39]:
OPTIONS = ["A", "B", "C", "D", "E"]

def split_prompt(prompt):
    text = str(prompt).strip()
    prefix, suffix = '', ''

    ci = text.find(': ')
    if ci != -1:
        prefix = text[:ci].strip()
        text = text[ci + 1:].strip()

    n_q = text.count('?')

    if n_q == 1:
        qi = text.index('?')
        suffix = text[qi + 1:].strip()
        text = text[:qi + 1].strip()

    elif n_q == 2:
        first_q = text.index('?')
        prefix = text[:first_q + 1].strip()

        text = text[first_q + 1:].strip()
        second_q = text.index('?')

        suffix = text[second_q + 1:].strip()
        text = text[:second_q + 1].strip()

    return (prefix or None), text, (suffix or None)


def clean(text, is_prompt=True):
    if(is_prompt):
        text = split_prompt(text)[1]
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def tokenize(text, is_prompt=True):
    return clean(text, is_prompt).split()


def ap_at_3(actual, predicted):
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0


def map_at_3(actuals, predictions):
    return float(np.mean([ap_at_3(a, p) for a, p in zip(actuals, predictions)]))


def make_submission(ids, predictions):
    return pd.DataFrame({
        'id': ids,
        'prediction': [' '.join(p[:3]) for p in predictions]
    })

In [40]:
df['q_clean'] = df['prompt'].apply(clean)

for opt in OPTIONS:
    df[f'{opt}_clean'] = df[opt].apply(lambda text: clean(text, is_prompt=False))

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       2000 non-null   int64 
 1   prompt   2000 non-null   object
 2   A        2000 non-null   object
 3   B        2000 non-null   object
 4   C        2000 non-null   object
 5   D        2000 non-null   object
 6   E        2000 non-null   object
 7   answer   2000 non-null   object
 8   q_clean  2000 non-null   object
 9   A_clean  2000 non-null   object
 10  B_clean  2000 non-null   object
 11  C_clean  2000 non-null   object
 12  D_clean  2000 non-null   object
 13  E_clean  2000 non-null   object
dtypes: int64(1), object(13)
memory usage: 218.9+ KB


In [41]:
print("Empty questions:", (df['q_clean'].str.len() == 0).sum())

for opt in OPTIONS:
    empty_count = (df[f'{opt}_clean'].str.len() == 0).sum()
    print(opt, empty_count)

Empty questions: 0
A 0
B 0
C 0
D 0
E 0


In [42]:
unique_qs = df['q_clean'].unique()
(len(df), len(unique_qs), len(set(unique_qs)))

(2000, 252, 252)

In [43]:
rng = np.random.default_rng(33)
rng.shuffle(unique_qs)

val_qs = set(unique_qs[:int(len(unique_qs)*0.20)])
train_qs = set(unique_qs[int(len(unique_qs)*0.20):])

train_df = df[df['q_clean'].isin(train_qs)].reset_index(drop=True)
val_df = df[df['q_clean'].isin(val_qs)].reset_index(drop=True)

In [44]:
def create_pair_dataframe(source_df):
    pair_rows = []

    for row in source_df.itertuples(index=False):
        question_tokens = row.q_clean.split()

        for opt in OPTIONS:
            option_text = getattr(row, f'{opt}_clean')
            option_tokens = option_text.split()

            tokens = question_tokens + ['<SEP>'] + option_tokens

            pair_rows.append({
                'id': row.id,
                'option': opt,
                'tokens': tokens,
                'length': len(tokens),
                'label': int(row.answer == opt)
            })

    return pd.DataFrame(pair_rows)

train_pairs = create_pair_dataframe(train_df)
val_pairs = create_pair_dataframe(val_df)

In [45]:
print(train_pairs.head(10))
train_pairs['length'].describe(percentiles=[0.50, 0.90, 0.95, 0.99])

   id option                                             tokens  length  label
0   1      A  [what, is, martin, heidegger, s, view, on, the...      67      0
1   1      B  [what, is, martin, heidegger, s, view, on, the...      56      1
2   1      C  [what, is, martin, heidegger, s, view, on, the...      53      0
3   1      D  [what, is, martin, heidegger, s, view, on, the...      46      0
4   1      E  [what, is, martin, heidegger, s, view, on, the...      46      0
5   4      A  [what, is, martin, heidegger, s, view, on, the...      67      0
6   4      B  [what, is, martin, heidegger, s, view, on, the...      56      1
7   4      C  [what, is, martin, heidegger, s, view, on, the...      53      0
8   4      D  [what, is, martin, heidegger, s, view, on, the...      46      0
9   4      E  [what, is, martin, heidegger, s, view, on, the...      46      0


count    7795.000000
mean       38.864785
std        19.038450
min        10.000000
50%        34.000000
90%        63.000000
95%        75.000000
99%        98.180000
max       149.000000
Name: length, dtype: float64

In [46]:
class Vocabulary:
    def __init__(self):
        self.t2id = {
            '<PAD>': 0,
            '<UNK>': 1,
            '<SEP>': 2
        }
        self.id2t = {
            0: '<PAD>',
            1: '<UNK>',
            2: '<SEP>'
        }

    def build(self, token_lists):
        counter = Counter()

        for tokens in token_lists:
            counter.update(tokens)

        for token, _ in counter.most_common():
            if token in self.t2id:
                continue
            idx = len(self.t2id)
            self.t2id[token] = idx
            self.id2t[idx] = token

    def encode(self, tokens, max_len):
        tokens = tokens[:max_len]
        ids = [self.t2id.get(token, 1) for token in tokens]

        real_length = len(ids)
        ids += [0] * (max_len - real_length)

        return ids, real_length

    def __len__(self):
        return len(self.t2id)

vocab = Vocabulary()
vocab.build(train_pairs['tokens'])

len(vocab)

2568

In [47]:
MAX_LEN=100

sample_tokens = train_pairs.iloc[0]['tokens']

ids, real_length = vocab.encode(sample_tokens, max_len=MAX_LEN)

print("Original tokens:", sample_tokens)
print("Encoded IDs:", ids[:real_length])

sep_position = sample_tokens.index('<SEP>')

print("SEP position:", sep_position)
print("SEP ID:", ids[sep_position])

decoded_tokens = [vocab.id2t[idx] for idx in ids[:real_length]]
decoded_tokens

Original tokens: ['what', 'is', 'martin', 'heidegger', 's', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', '<SEP>', 'martin', 'heidegger', 'believes', 'that', 'humans', 'exist', 'within', 'a', 'time', 'continuum', 'that', 'is', 'infinite', 'and', 'does', 'not', 'have', 'a', 'defined', 'beginning', 'or', 'end', 'the', 'relationship', 'to', 'the', 'past', 'involves', 'acknowledging', 'it', 'as', 'a', 'historical', 'era', 'and', 'the', 'relationship', 'to', 'the', 'future', 'involves', 'creating', 'a', 'world', 'that', 'will', 'endure', 'beyond', 'one', 's', 'own', 'time']
Encoded IDs: [8, 5, 439, 440, 17, 441, 25, 3, 40, 23, 30, 9, 519, 191, 2, 439, 440, 1146, 11, 1354, 429, 168, 6, 30, 1692, 11, 5, 788, 9, 99, 43, 56, 6, 214, 1512, 16, 1998, 3, 40, 10, 3, 803, 590, 1999, 22, 18, 6, 2000, 2001, 9, 3, 40, 10, 3, 956, 590, 418, 6, 1369, 11, 113, 2002, 2003, 58, 17, 1598, 30]
SEP position: 14
SEP ID: 2


['what',
 'is',
 'martin',
 'heidegger',
 's',
 'view',
 'on',
 'the',
 'relationship',
 'between',
 'time',
 'and',
 'human',
 'existence',
 '<SEP>',
 'martin',
 'heidegger',
 'believes',
 'that',
 'humans',
 'exist',
 'within',
 'a',
 'time',
 'continuum',
 'that',
 'is',
 'infinite',
 'and',
 'does',
 'not',
 'have',
 'a',
 'defined',
 'beginning',
 'or',
 'end',
 'the',
 'relationship',
 'to',
 'the',
 'past',
 'involves',
 'acknowledging',
 'it',
 'as',
 'a',
 'historical',
 'era',
 'and',
 'the',
 'relationship',
 'to',
 'the',
 'future',
 'involves',
 'creating',
 'a',
 'world',
 'that',
 'will',
 'endure',
 'beyond',
 'one',
 's',
 'own',
 'time']

In [48]:
def unk_stats(pair_df, vocab):
    total, unknown, pairs_with_unk = 0, 0, 0
    unk_counter = Counter()

    for tokens in pair_df['tokens']:
        has_unk = False
        for token in tokens:
            total += 1
            if token not in vocab.t2id:
                unknown += 1
                unk_counter[token] += 1
                has_unk = True
        pairs_with_unk += int(has_unk)

    return total, unknown, unk_counter, pairs_with_unk

total_tokens, unknown_tokens, unknown_counter, pairs_with_unk = unk_stats(val_pairs, vocab)

print("Validation UNK rate:", unknown_tokens / total_tokens)
print("Validation pair-level UNK %:", pairs_with_unk / len(val_pairs) * 100)
print("Unique unknown tokens:", len(unknown_counter))
print("Most common unknown tokens:", unknown_counter.most_common(20))

Validation UNK rate: 0.16912398689429212
Validation pair-level UNK %: 93.92290249433105
Unique unknown tokens: 394
Most common unknown tokens: [('fatty', 229), ('l', 220), ('acyl', 220), ('thylakoid', 220), ('membranes', 220), ('fusion', 209), ('disorder', 200), ('hysteresis', 190), ('b', 180), ('environmental', 150), ('alloys', 150), ('propagation', 150), ('memristors', 143), ('simultaneity', 140), ('isaac', 140), ('pinched', 140), ('curves', 140), ('reram', 140), ('terminal', 140), ('dragging', 135)]


In [49]:
def encode_pairs(pair_df, vocab, max_len, has_labels=True):
    samples = []

    for row in pair_df.itertuples(index=False):
        input_ids, length = vocab.encode(row.tokens, max_len)

        sample = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'length': torch.tensor(length, dtype=torch.long),
            'id': row.id,
            'option': row.option
        }

        if has_labels:
            sample['label'] = torch.tensor(row.label, dtype=torch.float)

        samples.append(sample)

    return samples

In [50]:
BATCH_SIZE = 64

train_samples = encode_pairs(train_pairs, vocab, 100, has_labels=True)
val_samples = encode_pairs(val_pairs, vocab, 100, has_labels=True)

train_loader = DataLoader(train_samples, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_samples, batch_size=BATCH_SIZE*2, shuffle=False)

In [51]:
class BiLSTMMCQ(nn.Module):
    def __init__(self, vocab_size, padding_idx, embed_dim=128, hidden_dim=128, num_layers=1, dropout=0.3):
        super().__init__()

        self.padding_idx = padding_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx)
        self.embedding_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=input_ids.size(1)
        )

        mask = (input_ids != self.padding_idx).unsqueeze(-1).to(output.dtype)
        pooled_output = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        return self.classifier(pooled_output).squeeze(-1)

In [52]:
model = BiLSTMMCQ(
    vocab_size=len(vocab),
    padding_idx=0,
    embed_dim=128,
    hidden_dim=128,
    num_layers=1,
    dropout=0.3
).to(device)

total_parameters = sum(parameter.numel() for parameter in model.parameters())

trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")

Total parameters: 626,177
Trainable parameters: 626,177


In [53]:
batch = next(iter(train_loader))

input_ids = batch['input_ids'].to(device)
lengths = batch['length']

with torch.no_grad():
    logits = model(input_ids, lengths)

print("Logits shape:", logits.shape)
print("Example logits:", logits[:5])

Logits shape: torch.Size([64])
Example logits: tensor([ 0.1386,  0.0199, -0.0148,  0.2117,  0.0706], device='cuda:0')


In [54]:
POS_WEIGHT = 4.0
LEARNING_RATE = 1e-3

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], dtype=torch.float32, device=device))
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

In [55]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_samples = 0.0, 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        lengths = batch['length']
        labels = batch['label'].to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

In [56]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    prediction_rows = []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        lengths = batch['length']
        labels = batch['label'].to(device)

        logits = model(input_ids, lengths)
        loss = criterion(logits, labels)
        probabilities = torch.sigmoid(logits).cpu().numpy()

        batch_size = input_ids.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        labels_cpu = labels.cpu().numpy()
        ids_cpu = batch['id'].cpu().numpy()
        options = batch['option']

        for question_id, option, probability, label in zip(ids_cpu, options, probabilities, labels_cpu):
            prediction_rows.append({
                'id': int(question_id),
                'option': option,
                'probability': float(probability),
                'label': int(label)
            })

    result_df = pd.DataFrame(prediction_rows)

    predictions, actual_answers = [], []

    for _, group in result_df.groupby('id', sort=False):
        ranked_options = group.sort_values('probability', ascending=False)['option'].tolist()
        correct_options = group.loc[group['label'] == 1, 'option'].tolist()

        if len(correct_options) != 1:
            raise ValueError("Each question must have exactly one correct option.")

        predictions.append(ranked_options[:3])
        actual_answers.append(correct_options[0])

    return {
        'loss': total_loss / total_samples,
        'map3': map_at_3(actual_answers, predictions),
        'predictions': predictions,
        'actual_answers': actual_answers,
        'result_df': result_df
    }

In [57]:
EPOCHS = 150
PATIENCE = 25

best_map3 = -1.0
best_epoch = 0
epochs_without_improvement = 0

history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    validation_result = evaluate(model, val_loader, criterion, device)

    val_loss = validation_result['loss']
    val_map3 = validation_result['map3']

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_map3': val_map3
    })

    print(f"Epoch {epoch}/{EPOCHS} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val MAP@3: {val_map3:.4f}")

    if val_map3 > best_map3:
        best_map3 = val_map3
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(model.state_dict(), 'best_bilstm.pt')
        print(f"Best model saved: MAP@3={best_map3:.4f}")
    else:
        epochs_without_improvement += 1
        # if epochs_without_improvement >= PATIENCE:
        #     print(f"Early stopping after epoch {epoch}.")
        #     break

print("\nTraining complete")
print("Best epoch:", best_epoch)
print("Best validation MAP@3:", best_map3)

Epoch 1/150 | Train loss: 1.0799 | Val loss: 1.1176 | Val MAP@3: 0.4664
Best model saved: MAP@3=0.4664
Epoch 2/150 | Train loss: 0.8754 | Val loss: 1.2906 | Val MAP@3: 0.4286
Epoch 3/150 | Train loss: 0.6348 | Val loss: 1.4967 | Val MAP@3: 0.4418
Epoch 4/150 | Train loss: 0.5024 | Val loss: 1.4318 | Val MAP@3: 0.4743
Best model saved: MAP@3=0.4743
Epoch 5/150 | Train loss: 0.4080 | Val loss: 1.9795 | Val MAP@3: 0.4913
Best model saved: MAP@3=0.4913
Epoch 6/150 | Train loss: 0.3303 | Val loss: 2.5064 | Val MAP@3: 0.5030
Best model saved: MAP@3=0.5030
Epoch 7/150 | Train loss: 0.2797 | Val loss: 2.1894 | Val MAP@3: 0.5045
Best model saved: MAP@3=0.5045
Epoch 8/150 | Train loss: 0.2465 | Val loss: 2.9011 | Val MAP@3: 0.5325
Best model saved: MAP@3=0.5325
Epoch 9/150 | Train loss: 0.1992 | Val loss: 3.2152 | Val MAP@3: 0.5004
Epoch 10/150 | Train loss: 0.1842 | Val loss: 3.4025 | Val MAP@3: 0.5246
Epoch 11/150 | Train loss: 0.1533 | Val loss: 4.1457 | Val MAP@3: 0.5204
Epoch 12/150 | Train

In [58]:
history_df = pd.DataFrame(history)
history_df

,epoch,train_loss,val_loss,val_map3
0,1,1.079946,1.117609,0.466364
1,2,0.875428,1.290609,0.428571
2,3,0.634776,1.496705,0.441799
3,4,0.502434,1.431811,0.474301
4,5,0.407961,1.979542,0.491308
...,...,...,...,...
145,146,0.010931,5.528565,0.500378
146,147,0.008028,6.092842,0.468254
147,148,0.006023,6.643611,0.501512
148,149,0.008786,6.239022,0.479214


In [59]:
model.load_state_dict(torch.load('best_bilstm.pt', map_location=device, weights_only=True))
model.eval()

BiLSTMMCQ(
  (embedding): Embedding(2568, 128, padding_idx=0)
  (embedding_dropout): Dropout(p=0.3, inplace=False)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [60]:
best_validation_result = evaluate(model, val_loader, criterion, device)
print("Best validation MAP@3:", best_validation_result['map3'])

validation_scores = best_validation_result['result_df'].copy()
validation_scores = validation_scores.sort_values(['id', 'probability'], ascending=[True, False])

validation_scores.head(20)

Best validation MAP@3: 0.5903250188964475


,id,option,probability,label
0,2,A,0.999965,1
4,2,E,0.999965,0
1,2,B,0.999962,0
3,2,D,0.000269,0
2,2,C,0.000232,0
5,3,A,0.000153,0
6,3,B,0.000153,0
7,3,C,0.000153,1
8,3,D,0.000153,0
9,3,E,0.000153,0


In [61]:
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
test_df['q_clean'] = test_df['prompt'].apply(clean)

for opt in OPTIONS:
    test_df[f'{opt}_clean'] = test_df[opt].apply(lambda text: clean(text, is_prompt=False))

print(test_df.shape)
test_df.head()

(500, 13)


,id,prompt,A,B,C,D,E,q_clean,A_clean,B_clean,C_clean,D_clean,E_clean
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",what is the relationship between the hamiltoni...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...",what is the estimated redshift of ceers 93316 ...,approximately z 6 0 corresponding to 1 billion...,approximately z 16 7 corresponding to 235 8 mi...,approximately z 3 0 corresponding to 5 billion...,approximately z 10 0 corresponding to 13 billi...,approximately z 13 0 corresponding to 30 billi...
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...,what is the reason for the sun appearing sligh...,the sun appears yellowish due to a reflection ...,the longer wavelengths of light such as red an...,the sun appears yellowish due to the scatterin...,the sun emits a yellow light due to its own sp...,the atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,what is the significance of the redshift dista...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...,observations of the redshift distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,what is the landau lifshitz gilbert equation u...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...,the landau lifshitz gilbert equation is a diff...


In [62]:
def create_test_pair_dataframe(source_df):
    pair_rows = []

    for row in source_df.itertuples(index=False):
        question_tokens = row.q_clean.split()

        for opt in OPTIONS:
            option_text = getattr(row, f'{opt}_clean')
            option_tokens = option_text.split()
            tokens = question_tokens + ['<SEP>'] + option_tokens

            pair_rows.append({
                'id': int(row.id),
                'option': opt,
                'tokens': tokens,
                'length': len(tokens)
            })

    return pd.DataFrame(pair_rows)

In [63]:
test_pairs = create_test_pair_dataframe(test_df)
test_samples = encode_pairs(test_pairs, vocab, 100, has_labels=False)
test_loader = DataLoader(test_samples, batch_size=BATCH_SIZE * 2, shuffle=False)

(len(test_df), len(test_pairs))

(500, 2500)

In [64]:
@torch.no_grad()
def predict_test(model, loader, device):
    model.eval()
    prediction_rows = []

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        lengths = batch['length']

        logits = model(input_ids, lengths)
        probabilities = torch.sigmoid(logits).cpu().numpy()

        ids_cpu = batch['id'].cpu().numpy()
        options = batch['option']

        for question_id, option, probability in zip(ids_cpu, options, probabilities):
            prediction_rows.append({
                'id': int(question_id),
                'option': option,
                'probability': float(probability)
            })

    result_df = pd.DataFrame(prediction_rows)

    submission_ids, predictions = [], []

    for question_id, group in result_df.groupby('id', sort=False):
        ranked_options = group.sort_values('probability', ascending=False)['option'].tolist()
        submission_ids.append(question_id)
        predictions.append(ranked_options[:3])

    return submission_ids, predictions, result_df


In [65]:
test_ids, test_predictions, test_scores = predict_test(model, test_loader, device)

In [66]:
submission = make_submission(test_ids, test_predictions)
submission.to_csv('submission.csv', index=False)

submission.head(5)

,id,prediction
0,1,A C D
1,2,D C B
2,3,B D C
3,4,E C A
4,5,D C E
